[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://githubtocolab.com/Soliman2020/python_48h_notebooks/blob/master/Session_08_Pandas_Foundations_and_Exploration.ipynb)

# Session 08 — Pandas Foundations and Exploration
## Python Fundamentals & Practical Applications — 48-Hour Edition

**STUDENT EDITION — WRITE YOUR OWN CODE**  
**Duration:** 4 classroom hours (including a 30-minute break)  
**Audience:** complete beginners and learners transferring from spreadsheets or other programming languages

> **Beginner promise:** We build one table step by step, predict results, practise each idea, and finish with a small CSV report using realistic synthetic data.
> **Working habit:** Ask a question first, inspect the data, then write the smallest useful Pandas expression.

In [ ]:
student_name = input("Enter your name: ")
print(f"{student_name}, welcome to the Pandas foundations session.")

## Learning outcomes

By the end of this session, you should be able to:

1. Explain the difference between a DataFrame and a Series.
2. Read a CSV file and inspect its shape, columns, types, and first rows.
3. Select columns and filter rows with Boolean conditions.
4. Create a calculated column.
5. Summarise categories with `value_counts()`, `describe()`, and `groupby()`.
6. Save a result as a CSV file.

**Total: 240 minutes.** Challenge work may become homework if the class needs more time.

## Google Colab quick start — ابدأ من هنا

Open the notebook in [Google Colab](https://colab.research.google.com/) or upload it using **File → Upload notebook**. Run the setup cell before the examples.

The course datasets are synthetic and are supplied in the course `datasets` folder.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

IS_COLAB = "google.colab" in sys.modules
if IS_COLAB:
    COURSE_ROOT = Path("/content/python_48h_course")
else:
    HERE = Path.cwd()
    COURSE_ROOT = HERE

DATA_DIR = COURSE_ROOT / "datasets"
OUTPUT_DIR = COURSE_ROOT / "outputs"
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)



print("Dataset folder:", DATA_DIR)
print("Output folder:", OUTPUT_DIR)

* Visit [downgit page](https://downgit.github.io/#/home) , paste the target dataset folder URL: `https://github.com/Soliman2020/python_48h_notebooks/tree/master/datasets`, then click on `Download` button.
* On your PC, after downloading the zip file, Unzip it and upload the inner files to `dataset folder` created its directory before on the previous code cell. 

In [ ]:
REQUIRED_DATA_FILES = [
    "work_items.csv",
    "energy_readings.csv",
    "transactions_synthetic.csv",
]
missing_files = [name for name in REQUIRED_DATA_FILES if not (DATA_DIR / name).is_file()]
if missing_files:
    raise FileNotFoundError(
        "Teaching datasets are missing from "
        f"{DATA_DIR}. Add the course datasets folder, then run this cell again. "
        f"Missing: {', '.join(missing_files)}")
    
print("Required datasets found:", len(REQUIRED_DATA_FILES))

## 1. A DataFrame is a programmable table

A **DataFrame** has rows and columns. A single column is usually a **Series**. The labels help us ask for the data we need.

Run the example, then change one value and run it again.

In [ ]:
# Create a sample DataFrame
sample = pd.DataFrame({
    "team": ["Ahly", "Zamalek", "Pyramids"],
    "score": [82, 74, 91]
})

sample

In [ ]:
print("Type of sample:", type(sample).__name__)

print("Type of one column:", type(sample["score"]).__name__)

In [ ]:
# You can either add a row or change a score like this:

sample.loc[3] = ["PSG", 94]  # Add a new row

sample.loc[1, "score"] = 80  # Change the score for the Green team

sample

### Guided activity 1 — create and inspect a table

Before running the next cell, predict its shape and column names. Then add extra row to `sample` and change one score.

In [ ]:
# Get information about the DataFrame Shape and columns

print("Shape:", sample.shape)

print("Columns:", sample.columns.to_list())

print("Scores:")

print(sample["score"]) 

In [ ]:
# TODO: Add one row to `sample` or change one score.





## 2. Read a CSV and inspect before analysing

The first step with an unfamiliar table is not a chart. It is a quick inspection: size, names, sample rows, data types, and missing values.

`work_items.csv` contains synthetic operational records used throughout this session.

In [ ]:
# Read the work_items.csv file into a DataFrame
work = pd.read_csv(DATA_DIR / "work_items.csv")

print("Shape:", work.shape)

print("Columns:", work.columns.to_list())

work.head()

In [ ]:
# Get information about the DataFrame
work.info()

In [ ]:
# Get descriptive statistics for the specified columns
work[["hours_open", "estimated_cost"]].describe().round(2)

### Guided activity 2 — inspect a different dataset

Load `energy_readings.csv`. Print its shape, columns, and first three rows. Which column looks numeric? Which column identifies a group such as a site?

In [ ]:
# TODO: load energy_readings.csv and inspect it




## 3. Select columns and filter rows

Use brackets to select columns. Use `.loc[rows, columns]` when you want to make the row and column choices explicit. A condition creates a Boolean Series: `True` keeps a row and `False` excludes it.

**When combining conditions, put parentheses around each comparison.**

In [ ]:
# Select specific columns
selected = work[["case_id", "sector", "status", "priority"]]

selected.head()

In [ ]:
# Sample 5 rows from the selected DataFrame
selected.sample(5, random_state=42)

In [ ]:
# Filter the DataFrame based on status and priority

is_open = work["status"] == "open"

is_medium = work["priority"] == "medium"

# Create a new DataFrame with open and medium priority cases and select specific columns with copy
open_medium = work.loc[(is_open) & (is_medium), ["case_id", "status", "priority", "sector", "hours_open"]].copy()

# Display the first 5 rows of the new DataFrame
open_medium.head()

In [ ]:
# Filter the work DataFrame to keep only cases that are:
# - currently open
# - assigned medium priority using the query method and select specific columns with copy

open_medium = (
    work.query("status == 'open' and priority == 'medium'")
    # Select only the columns needed for this analysis
    .loc[:, ["case_id", "status", "priority", "sector", "hours_open"]]
    # Create an independent copy so later changes do not modify the original DataFrame
    .copy()
)

# Show the first 5 filtered cases for a quick check
open_medium.head()

- Use `.copy()` if you plan to modify `open_medium` afterward. It creates an **independent DataFrame**, so changes do not affect the original work DataFrame.

In [ ]:
# Get the number of rows in the new DataFrame
open_medium.shape[0]

In [ ]:
# Get the number of rows in the new DataFrame / 2nd method
len(open_medium)

### Selecting by label: `.loc` versus position: `.iloc`

Pandas provides two clear ways to select rows and columns:

- `.loc` uses **labels**, such as row labels and column names.
- `.iloc` uses **integer positions**, starting at zero.

Use `.loc` when the meaning of the label matters. Use `.iloc` when you deliberately want a position.

| Goal | Example |
|---|---|
| First three rows by position | `work.iloc[:3]` |
| Rows 2 through 4 by position | `work.iloc[2:5]` |
| All rows, selected columns by name | `work.loc[:, ["case_id", "sector"]]` |
| Rows matching a condition and selected columns | `work.loc[is_open, ["case_id", "status"]]` |

The end of an `.iloc` slice is excluded, as in normal Python slicing. For a label slice, `.loc` includes both endpoint labels when they exist.

In [ ]:
print("First three rows and first four columns by position (.iloc):")
print(work.iloc[:3, :4])

In [ ]:
print("First three rows and named columns by label (.loc):")
print(work.loc[:2, ["case_id", "sector", "team", "priority"]])

In [ ]:
print("One value by position:", work.iloc[0, 1])

In [ ]:
print("One value by labels:", work.loc[0, "sector"])

In [ ]:
# TODO: use describe() for hours_open and estimated_cost
# Then write your answers in comments.

### Guided activity 2b — practise `describe()`

Use `describe()` to summarise `hours_open` and `estimated_cost`. Then answer in a comment: which column has the larger average? What does the median tell you about a typical record?

### Guided activity 3 — build a filter in stages

Filter `work` to records from one sector. First inspect `work["sector"].value_counts()`, then create a condition, then print the matching case IDs.

In [ ]:
work["sector"].value_counts()

In [ ]:
# TODO: choose a sector, filter the rows, and print case_id





## 4. Create a calculated column

A calculated column turns a raw field into a useful measure. Pandas applies the operation to the whole column.

Use `.copy()` when you create a filtered table that you intend to modify, so the original table remains unchanged.

In [ ]:
# Add a new column to the DataFrame that calculates the age of each case in days
work["age_days"] = work["hours_open"] / 24

# Display the first 5 rows of the DataFrame with the new column
work[["case_id", "hours_open", "age_days"]].head()


## 5. Summarise categories with `groupby()`

`value_counts()` is a convenient first step when the question is simply “how many records are in each category?”.

`groupby()` is useful when we want a summary calculation for every category. We will build the idea from beginner to advanced.

### Groupby level 1 — count records

Question: how many cases belong to each sector?

The result has one value for each sector.

In [ ]:
cases_by_sector = work.groupby("sector")["case_id"].count()
cases_by_sector

### Groupby level 2 — calculate one average

Question: what is the average number of hours open in each sector?

We select one numeric column and apply one aggregation function.

In [ ]:
average_hours_by_sector = work.groupby("sector")["hours_open"].mean()
average_hours_by_sector.sort_values(ascending=False).round(2)

### Groupby level 3 — calculate several summaries

Once one summary works, we can calculate a count and an average together. Named aggregations give the result useful column names.

In [ ]:
sector_summary = (
    work.groupby("sector")
        .agg(
            cases=("case_id", "count"),
            average_hours=("hours_open", "mean")
        )
        .sort_values("average_hours", ascending=False)
)
sector_summary.round(1)

### Groupby level 4 — two grouping columns (challenge)

Question: how many cases are in each sector and priority combination?

The result has two grouping levels. `reset_index()` turns those levels back into ordinary columns, which is often easier to read and export.

In [ ]:
cases_by_sector_priority = (
    work.groupby(["sector", "priority"])
        .size()
        .reset_index(name="cases")
        .sort_values("cases", ascending=False)
)
cases_by_sector_priority.head(10)

### Guided activity 4 — compare groups

Create a summary by `priority` with the number of cases and average `estimated_cost`. Sort it from largest average cost to smallest.

In [ ]:
# TODO: use groupby().agg() and sort_values()


## Practice tasks

Complete the guided activities first. Then work independently. Explain what each result means in plain English, not only what code produced it.



### Applied task A — `S08-E01`
Load `transactions_synthetic.csv`. Print its shape, column names, and the five most common transaction types.

In [ ]:
# TODO:




### Applied task B — `S08-E02`
Using `work_items.csv`, select high-priority records and print their case IDs and estimated costs.

In [ ]:
# TODO:




### Challenge — `S08-E03`
Create a summary by both `sector` and `priority`, with case count and average `hours_open`. Sort by case count descending.

In [ ]:
# TODO:




## Guided mini-project — Sector performance snapshot

Create a readable CSV report from `work_items.csv`. Your report must contain one row per sector and:

- case count
- median `hours_open`
- average `estimated_cost`

Save it as `session_08_sector_snapshot.csv` in `OUTPUT_DIR`. Write one sentence explaining an observation from your report.

In [ ]:
# Save the already-created sector summary as a small example CSV
example_path = OUTPUT_DIR / "example_sector_summary.csv"
sector_summary.round(2).to_csv(example_path)
print("Saved:", example_path)
print(pd.read_csv(example_path).head())

### Worked example — save a result as CSV

`to_csv()` writes a DataFrame to a CSV file. `index=False` prevents the Pandas row index from becoming an extra column in the file.

In [ ]:
# TODO: group, aggregate, sort, and export the report
# Hint: finish with .to_csv(
#     OUTPUT_DIR / "session_08_sector_snapshot.csv",
#     index=False
# )

## Exit check — answer from memory

1. What is the difference between a DataFrame and a Series?
2. Which commands help you inspect a CSV after reading it?
3. What is the difference between `.loc` and `.iloc`?
4. How do you create a calculated column?
5. When would you use `value_counts()` instead of `groupby()`?
6. What question does `groupby()` help answer?
7. How do you save a DataFrame as a CSV?

## Common mistakes and next steps

- Inspect the column names before typing them.
- Use `==` for comparison and `=` for assignment.
- Remember that `.loc` uses labels and `.iloc` uses positions.
- Use parentheses when combining Boolean conditions.
- Do not assume the first five rows represent the whole dataset.
- Explain a grouped result in words.
- Check that an exported file exists and contains the expected columns.

**Next session:** we will clean imperfect data, handle missing values, convert dates, combine tables, and build a report.